# 91 News Radius Context

In [ ]:
import os, re, json, hashlib, platform, sys, math
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
import numpy as np
import yaml

PROJECT_ROOT = Path.cwd().resolve().parents[0] if (Path.cwd().name == "notebooks") else Path.cwd().resolve()
DATA = PROJECT_ROOT / "data_sources"
OUTPUTS = PROJECT_ROOT / "outputs"
CONFIGS = PROJECT_ROOT / "configs"
OUTPUTS.mkdir(exist_ok=True)

def utcnow():
    return datetime.now(timezone.utc).isoformat()

def sha256_file(p: Path) -> str:
    h = hashlib.sha256()
    with p.open("rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

def write_json(p: Path, obj):
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(json.dumps(obj, indent=2, sort_keys=True), encoding="utf-8")

def load_yaml(p: Path):
    return yaml.safe_load(p.read_text(encoding="utf-8")) if p.exists() else {}

def safe_read_csv(path):
    p = Path(path)
    if not p.is_absolute():
        p = PROJECT_ROOT / p
    if not p.exists():
        return pd.DataFrame(), {"path": str(p), "status": "missing"}
    try:
        if p.stat().st_size == 0:
            return pd.DataFrame(), {"path": str(p), "status": "empty_file"}
        df = pd.read_csv(p)
        if df.empty:
            return pd.DataFrame(), {"path": str(p), "status": "no_rows", "sha256": sha256_file(p)}
        return df, {"path": str(p), "status": "ok", "sha256": sha256_file(p), "rows": int(len(df))}
    except Exception as e:
        return pd.DataFrame(), {"path": str(p), "status": f"error:{e}"}

def manifest_base(step_name: str, config_paths=None):
    return {"project": "AirQuality26", "step": step_name, "created_at_utc": utcnow(), "platform": {"python": sys.version, "platform": platform.platform()}, "configs": {str(p): sha256_file(Path(p)) for p in (config_paths or []) if Path(p).exists()}, "artifacts": [], "notes": []}

def add_artifact(manifest, p: Path):
    if p.exists():
        manifest["artifacts"].append({"path": str(p), "sha256": sha256_file(p)})

def haversine_miles(lat1, lon1, lat2, lon2):
    vals = [lat1, lon1, lat2, lon2]
    if any(pd.isna(vals)):
        return np.nan
    r = 3958.7613
    p1, p2 = math.radians(float(lat1)), math.radians(float(lat2))
    dphi = math.radians(float(lat2) - float(lat1))
    dlmb = math.radians(float(lon2) - float(lon1))
    a = math.sin(dphi/2)**2 + math.cos(p1)*math.cos(p2)*math.sin(dlmb/2)**2
    return 2*r*math.asin(math.sqrt(a))

phase_geo = load_yaml(CONFIGS / "phase88_94_geospatial.yml")

In [ ]:
step = "91_news_radius_context"
out = OUTPUTS / step
out.mkdir(parents=True, exist_ok=True)

sites, _ = safe_read_csv(OUTPUTS / "88_site_coordinate_registry" / "site_coordinate_registry.csv")
geo_cfg = load_yaml(CONFIGS / "site_geospatial_registry.yml")
incident_terms = phase_geo.get("incident_terms", ["fire","odour","smoke","pollution","complaint","ash","emissions","breach","shutdown"])

rows = []
for _, s in sites.iterrows():
    spec = (geo_cfg.get("sites", {}) or {}).get(str(s.get("site_key")), {})
    nearby = []
    nearby.extend(spec.get("nearby_towns", []) or [])
    if str(s.get("nearest_town","")).strip():
        nearby.append(str(s.get("nearest_town")))
    if str(s.get("nearest_city","")).strip():
        nearby.append(str(s.get("nearest_city")))
    nearby = sorted(set([x for x in nearby if str(x).strip()]))
    search_terms = [str(s.get("facility_name","")), str(s.get("permit_id",""))] + nearby + incident_terms
    rows.append({
        "site_key": s.get("site_key"),
        "permit_id": s.get("permit_id",""),
        "facility_name": s.get("facility_name",""),
        "nearest_town": s.get("nearest_town",""),
        "nearest_city": s.get("nearest_city",""),
        "nearby_places": "|".join(nearby),
        "suggested_news_query_terms": " | ".join([x for x in search_terms if str(x).strip()]),
    })

news_ctx = pd.DataFrame(rows)
news_ctx.to_csv(out / "news_radius_context.csv", index=False)
write_json(out / "news_radius_context_summary.json", {"rows": int(len(news_ctx))})
manifest = manifest_base(step, [CONFIGS / "phase88_94_geospatial.yml", CONFIGS / "site_geospatial_registry.yml"])
add_artifact(manifest, out / "news_radius_context.csv")
add_artifact(manifest, out / "news_radius_context_summary.json")
write_json(out / "manifest.json", manifest)
display(news_ctx.head(20))